# ISLES'24 pilot: raw 4D CTP vs perfusion maps (small sample)

**Purpose:** a cheap pilot run to get rough numbers, not a result for publication.
1. Check that the pipeline runs end to end and that raw CTP, maps and the lesion mask share one voxel grid.
2. Estimate the per-patient spread of Δ Dice (σΔ), so the full-size experiment can be planned.
3. Get a first look at whether raw beats maps, and whether the N1/N2 controls remove that gain.

**Arms** (same 2D U-Net and recipe; only the first conv layer differs in input channels):
| Arm | Input channels |
|---|---|
| `maps` | NCCT, CTA, CBF, CBV, MTT, Tmax |
| `raw`  | NCCT, CTA, T CTP frames (peak-aligned window, baseline-subtracted) |
| `N1`   | NCCT, CTA, temporal mean, temporal max (temporal axis destroyed) |
| `N2`   | NCCT, CTA, the same T frames in a random order fixed per patient (temporal order destroyed) |

**Not** nnU-Net, downsampled in-plane, 2D, small n. Treat every number as a pilot estimate.

**Kaggle settings:** Accelerator = GPU T4, Internet = On. Expected runtime with the default config: under 1.5 h (estimate).

Data: Hugging Face mirror `hugging-science/isles24-stroke` (one parquet shard per patient, CC BY-NC-SA 4.0).
Cite the ISLES'24 dataset and challenge papers if you use the results.

In [ ]:
# ---------------- CONFIG ----------------
import os

CFG = dict(
    repo_id="hugging-science/isles24-stroke",
    n_total_shards=149,
    n_cases=40,            # pilot size; raise to 60-80 if time allows
    seed=2027,
    t_window=16,           # CTP frames kept around the bolus peak
    t_pre=5,               # frames kept before the peak
    n_baseline=3,          # first frames averaged as pre-contrast baseline
    img_size=128,          # in-plane size after downsampling
    min_brain_frac=0.02,   # keep slices with at least this brain fraction
    n_folds=5,
    epochs=40,
    batch_size=16,
    lr=1e-3,
    base_ch=32,
    lesion_slice_frac=0.5, # share of training batches drawn from lesion slices
    n_boot=10000,
    arms=["maps", "raw", "N1", "N2"],
    work_dir=os.environ.get("PILOT_WORK", "/kaggle/working"),
    cache_dir=os.environ.get("PILOT_CACHE", "/kaggle/temp/cache"),
    local_shard_dir=os.environ.get("PILOT_LOCAL_SHARDS", ""),  # for offline testing only
)
# quick overrides for smoke tests, e.g. PILOT_OVERRIDES="n_cases=6,epochs=1,img_size=64,n_folds=3"
for kv in filter(None, os.environ.get("PILOT_OVERRIDES", "").split(",")):
    k, v = kv.split("=")
    CFG[k] = type(CFG[k])(v)
print(CFG)

In [ ]:
import subprocess, sys
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
try:
    import nibabel  # noqa
except ImportError:
    _pip("nibabel")
try:
    import huggingface_hub  # noqa
except ImportError:
    _pip("huggingface_hub")

In [ ]:
import gc, io, json, gzip, tempfile, time, random
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import nibabel as nib
from scipy import ndimage

os.makedirs(CFG["work_dir"], exist_ok=True)
os.makedirs(CFG["cache_dir"], exist_ok=True)
rng = np.random.default_rng(CFG["seed"])
random.seed(CFG["seed"])

## 1. Download and preprocess a random subset of patients
Each shard is one patient (about 150-260 MB). Each shard is preprocessed to a small `.npz`, then deleted.

In [ ]:
def fetch_shard(idx):
    name = f"train-{idx:05d}-of-{CFG['n_total_shards']:05d}.parquet"
    if CFG["local_shard_dir"]:
        return os.path.join(CFG["local_shard_dir"], name), False
    from huggingface_hub import hf_hub_download
    token = os.environ.get("HF_TOKEN")
    try:  # Kaggle secret, optional
        from kaggle_secrets import UserSecretsClient
        token = token or UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    p = hf_hub_download(CFG["repo_id"], filename=f"data/{name}", repo_type="dataset",
                        cache_dir=CFG["cache_dir"], token=token)
    return p, True


TMP_FILES = []


def to_nifti(obj):
    """Decode one NIfTI cell from the parquet row (dict with bytes, or raw bytes).
    The image is written uncompressed to disk and memory-mapped, so a large 4D CTP
    never has to sit in RAM at full size."""
    if obj is None:
        return None
    if isinstance(obj, dict):
        b = obj.get("bytes")
        if b is None and obj.get("path") and os.path.exists(obj["path"]):
            return nib.load(obj["path"])
    else:
        b = obj
    if b is None:
        return None
    if b[:2] == b"\x1f\x8b":
        b = gzip.decompress(b)
    with tempfile.NamedTemporaryFile(suffix=".nii", delete=False, dir=CFG["cache_dir"]) as f:
        f.write(b)
        path = f.name
    TMP_FILES.append(path)
    return nib.load(path, mmap=True)


def cleanup_tmp():
    while TMP_FILES:
        p = TMP_FILES.pop()
        if os.path.exists(p):
            os.remove(p)


def area_down(x, size, is_mask=False):
    """x: (Z, H, W) -> (Z, size, size) by zoom (linear, or nearest for masks)."""
    z, h, w = x.shape
    order = 0 if is_mask else 1
    return ndimage.zoom(x, (1, size / h, size / w), order=order, prefilter=False)


def brain_mask_from_ncct(ncct_zyx):
    m = (ncct_zyx > 0) & (ncct_zyx < 100)
    m = ndimage.binary_opening(m, iterations=2)
    out = np.zeros_like(m)
    for z in range(m.shape[0]):  # largest component per slice, holes filled
        lab, n = ndimage.label(m[z])
        if n == 0:
            continue
        sizes = ndimage.sum(m[z], lab, range(1, n + 1))
        out[z] = ndimage.binary_fill_holes(lab == (np.argmax(sizes) + 1))
    return out


def xyz_to_zyx(a):
    return np.transpose(a, (2, 1, 0)) if a.ndim == 3 else np.transpose(a, (3, 2, 1, 0))  # 4D -> (T, Z, Y, X)


def preprocess_case(path):
    tbl = pq.read_table(path)
    row = tbl.slice(0, 1).to_pylist()[0]
    sid = row.get("subject_id", os.path.basename(path))
    imgs = {k: to_nifti(row.get(k)) for k in ["ncct", "cta", "ctp", "cbf", "cbv", "mtt", "tmax", "lesion_mask"]}
    del row, tbl
    missing = [k for k, v in imgs.items() if v is None]
    if missing:
        return None, {"subject_id": sid, "status": f"missing {missing}"}

    shapes = {k: tuple(v.shape) for k, v in imgs.items()}
    ref = shapes["lesion_mask"]
    bad = [k for k, s in shapes.items() if s[:3] != ref]
    if bad:
        return None, {"subject_id": sid, "status": f"grid mismatch {bad}", **{f"shape_{k}": str(s) for k, s in shapes.items()}}

    zooms = imgs["lesion_mask"].header.get_zooms()[:3]
    f32 = lambda k: xyz_to_zyx(np.asarray(imgs[k].dataobj, dtype=np.float32))
    ncct, cta = f32("ncct"), f32("cta")
    maps = {k: f32(k) for k in ["cbf", "cbv", "mtt", "tmax"]}
    mask = xyz_to_zyx(np.asarray(imgs["lesion_mask"].dataobj) > 0.5)

    brain = brain_mask_from_ncct(ncct)
    keep = brain.reshape(brain.shape[0], -1).mean(1) >= CFG["min_brain_frac"]
    if keep.sum() == 0:
        return None, {"subject_id": sid, "status": "no brain slices"}
    sl = np.where(keep)[0]
    S = CFG["img_size"]

    # CTP, read frame by frame from the memory map: baseline-subtracted enhancement, peak-aligned window
    ctp_px = imgs["ctp"].dataobj
    T = ctp_px.shape[3]
    frame = lambda t: xyz_to_zyx(np.asarray(ctp_px[..., t], dtype=np.float32))
    base = np.mean([frame(t) for t in range(CFG["n_baseline"])], axis=0)
    curve = np.array([(frame(t) - base)[brain].mean() for t in range(T)])
    peak = int(np.argmax(curve))
    start = peak - CFG["t_pre"]
    idx = np.clip(np.arange(start, start + CFG["t_window"]), 0, T - 1)  # edge padding if near the ends
    win = np.stack([area_down(np.clip(frame(t) - base, -20, 150)[sl] / 100.0, S).astype(np.float16) for t in idx])
    del base, ctp_px, imgs
    gc.collect()

    # maps: Tmax and MTT in seconds (fixed scale); CBF/CBV relative to the case's brain median (rCBF-like)
    def rel(x):
        med = np.median(x[brain & (x > 0)]) if np.any(brain & (x > 0)) else 1.0
        return np.clip(x / max(med, 1e-6), 0, 3) / 3.0
    m_out = np.stack([rel(maps["cbf"]), rel(maps["cbv"]),
                      np.clip(maps["mtt"], 0, 20) / 20.0, np.clip(maps["tmax"], 0, 20) / 20.0])

    ncct_n = np.clip(ncct, 0, 80) / 80.0
    cta_n = np.clip(cta, -100, 400) / 500.0 + 0.2

    down = lambda a: area_down(a[sl], S).astype(np.float16)
    out = dict(
        ncct=down(ncct_n), cta=down(cta_n),
        maps=np.stack([down(m) for m in m_out]),
        ctp=win,
        brain=area_down(brain[sl].astype(np.float32), S, is_mask=True).astype(bool),
        mask=area_down(mask[sl].astype(np.float32), S, is_mask=True).astype(bool),
    )
    vox_ml = float(np.prod(zooms)) / 1000.0
    scale = (mask.shape[1] / S) * (mask.shape[2] / S)
    meta = {"subject_id": sid, "status": "ok", "T": T, "peak": peak, "n_slices": int(len(sl)),
            "gt_ml_full": float(mask.sum() * vox_ml), "vox_ml_down": vox_ml * scale,
            "shape": str(shapes["lesion_mask"]), "zooms": str(tuple(round(float(z), 3) for z in zooms))}
    return out, meta


chosen = sorted(rng.choice(CFG["n_total_shards"], size=CFG["n_cases"], replace=False).tolist())
print("shards:", chosen)

data, metas = {}, []
t0 = time.time()
for k, idx in enumerate(chosen):
    try:
        p, downloaded = fetch_shard(idx)
        out, meta = preprocess_case(p)
        if downloaded and os.path.exists(p):
            os.remove(os.path.realpath(p))
    except Exception as e:  # keep going; report at the end
        out, meta = None, {"subject_id": f"shard{idx}", "status": f"error {type(e).__name__}: {e}"}
    finally:
        cleanup_tmp()
    meta["shard"] = idx
    metas.append(meta)
    if out is not None:
        data[meta["subject_id"]] = (out, meta)
    print(f"[{k+1}/{len(chosen)}] shard {idx}: {meta['status']}  ({time.time()-t0:.0f}s)")
    gc.collect()

meta_df = pd.DataFrame(metas)
meta_df.to_csv(os.path.join(CFG["work_dir"], "cases.csv"), index=False)
print(meta_df["status"].value_counts())
assert len(data) >= CFG["n_folds"] * 2, (
    "Too few usable cases. If the status says 'grid mismatch', the mirror's CTP/maps are in native CTP space "
    "and the lesion mask is in NCCT space: use the Zenodo 'derivatives/*space-ncct*' files instead.")

Sanity check: the mean enhancement curve of a few patients should show one clear bolus peak.

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 4, figsize=(14, 3))
    sid0 = next(iter(data))
    d0, m0 = data[sid0]
    z = int(np.argmax(d0["mask"].sum((1, 2)))) if d0["mask"].any() else d0["mask"].shape[0] // 2
    axes[0].imshow(d0["ncct"][z].astype(np.float32), cmap="gray"); axes[0].set_title(f"{sid0} NCCT")
    axes[1].imshow(d0["maps"][3, z].astype(np.float32)); axes[1].set_title("Tmax")
    axes[2].imshow(d0["ctp"][CFG["t_pre"], z].astype(np.float32)); axes[2].set_title("CTP at peak")
    axes[3].imshow(d0["mask"][z]); axes[3].set_title("lesion mask")
    for a in axes: a.axis("off")
    plt.savefig(os.path.join(CFG["work_dir"], "sanity.png"), dpi=80, bbox_inches="tight"); plt.show()
except Exception as e:
    print("plot skipped:", e)

## 2. Build inputs per arm

In [ ]:
SIDS = sorted(data)
perm_rng = np.random.default_rng(CFG["seed"] + 1)
N2_PERM = {sid: perm_rng.permutation(CFG["t_window"]) for sid in SIDS}  # fixed per patient


def arm_input(arm, d, sid):
    base = [d["ncct"][None], d["cta"][None]]
    if arm == "maps":
        x = np.concatenate(base + [d["maps"]])
    elif arm == "raw":
        x = np.concatenate(base + [d["ctp"]])
    elif arm == "N1":
        c = d["ctp"].astype(np.float32)
        x = np.concatenate(base + [c.mean(0, keepdims=True).astype(np.float16), c.max(0, keepdims=True).astype(np.float16)])
    elif arm == "N2":
        x = np.concatenate(base + [d["ctp"][N2_PERM[sid]]])
    else:
        raise ValueError(arm)
    return np.transpose(x, (1, 0, 2, 3))  # (Z, C, H, W)


N_CH = {"maps": 6, "raw": 2 + CFG["t_window"], "N1": 4, "N2": 2 + CFG["t_window"]}

# patient-level folds, identical for every arm
order = np.random.default_rng(CFG["seed"] + 2).permutation(len(SIDS))
FOLDS = [[SIDS[i] for i in order[f::CFG["n_folds"]]] for f in range(CFG["n_folds"])]
json.dump(FOLDS, open(os.path.join(CFG["work_dir"], "folds.json"), "w"), indent=1)
print([len(f) for f in FOLDS])

## 3. Model and training (identical for all arms)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")


def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.InstanceNorm2d(o, affine=True), nn.LeakyReLU(0.01, True),
                         nn.Conv2d(o, o, 3, padding=1, bias=False), nn.InstanceNorm2d(o, affine=True), nn.LeakyReLU(0.01, True))


class UNet2D(nn.Module):
    def __init__(self, in_ch, b=32):
        super().__init__()
        ch = [b, b * 2, b * 4, b * 8]
        self.enc = nn.ModuleList([block(in_ch, ch[0]), block(ch[0], ch[1]), block(ch[1], ch[2]), block(ch[2], ch[3])])
        self.up = nn.ModuleList([nn.ConvTranspose2d(ch[i + 1], ch[i], 2, 2) for i in range(3)])
        self.dec = nn.ModuleList([block(ch[i] * 2, ch[i]) for i in range(3)])
        self.out = nn.Conv2d(ch[0], 1, 1)

    def forward(self, x):
        skips = []
        for i, e in enumerate(self.enc):
            x = e(x)
            if i < 3:
                skips.append(x)
                x = F.max_pool2d(x, 2)
        for i in reversed(range(3)):
            x = self.dec[i](torch.cat([self.up[i](x), skips[i]], 1))
        return self.out(x)


for a in CFG["arms"]:
    print(a, "params:", sum(p.numel() for p in UNet2D(N_CH[a], CFG["base_ch"]).parameters()))


def loss_fn(logit, y):
    p = torch.sigmoid(logit)
    inter = (p * y).sum((1, 2, 3))
    dice = 1 - (2 * inter + 1) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1)
    return F.binary_cross_entropy_with_logits(logit, y) + dice.mean()


def make_arrays(arm, sids):
    xs, ys, les = [], [], []
    for sid in sids:
        d, _ = data[sid]
        x = arm_input(arm, d, sid)
        xs.append(x); ys.append(d["mask"][:, None]); les.append(d["mask"].reshape(len(x), -1).any(1))
    return np.concatenate(xs), np.concatenate(ys), np.concatenate(les)


def train_one(arm, train_sids, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    X, Y, L = make_arrays(arm, train_sids)
    pos, allidx = np.where(L)[0], np.arange(len(X))
    model = UNet2D(N_CH[arm], CFG["base_ch"]).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-4)
    steps_per_epoch = max(1, len(X) // CFG["batch_size"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, CFG["lr"], total_steps=CFG["epochs"] * steps_per_epoch)
    scaler = torch.amp.GradScaler(enabled=(DEV == "cuda"))
    bs, nb_pos = CFG["batch_size"], int(CFG["batch_size"] * CFG["lesion_slice_frac"]) if len(pos) else 0
    model.train()
    for ep in range(CFG["epochs"]):
        for _ in range(steps_per_epoch):
            b = np.concatenate([np.random.choice(pos, nb_pos), np.random.choice(allidx, bs - nb_pos)])
            x = torch.from_numpy(X[b].astype(np.float32)).to(DEV)
            y = torch.from_numpy(Y[b].astype(np.float32)).to(DEV)
            if np.random.rand() < 0.5:
                x, y = x.flip(-1), y.flip(-1)
            with torch.autocast(device_type="cuda", enabled=(DEV == "cuda")):
                logit = model(x)
            loss = loss_fn(logit.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
    return model


@torch.no_grad()
def eval_patients(model, arm, sids):
    model.eval()
    rows = []
    for sid in sids:
        d, meta = data[sid]
        x = torch.from_numpy(arm_input(arm, d, sid).astype(np.float32))
        probs = []
        for i in range(0, len(x), 32):
            xb = x[i:i + 32].to(DEV)
            p = (torch.sigmoid(model(xb)) + torch.sigmoid(model(xb.flip(-1))).flip(-1)) / 2  # flip TTA
            probs.append(p.cpu())
        pred = (torch.cat(probs)[:, 0].numpy() > 0.5) & d["brain"]
        gt = d["mask"]
        inter, s = (pred & gt).sum(), pred.sum() + gt.sum()
        dice = 1.0 if s == 0 else 2 * inter / s
        v = meta["vox_ml_down"]
        rows.append({"subject_id": sid, "arm": arm, "dice": float(dice),
                     "gt_ml": float(gt.sum() * v), "pred_ml": float(pred.sum() * v),
                     "abs_vol_err_ml": float(abs(pred.sum() - gt.sum()) * v)})
    return rows

## 4. Run all arms × folds (results are saved after every run, so a cut session can resume)

In [ ]:
res_path = os.path.join(CFG["work_dir"], "per_patient.csv")
done = pd.read_csv(res_path) if os.path.exists(res_path) else pd.DataFrame()
done_keys = set(zip(done.get("arm", []), done.get("fold", [])))
all_rows = done.to_dict("records")

for f, val in enumerate(FOLDS):
    train = [s for s in SIDS if s not in val]
    for arm in CFG["arms"]:
        if (arm, f) in done_keys:
            continue
        t0 = time.time()
        model = train_one(arm, train, CFG["seed"] + 100 * f)  # same seed per fold across arms
        rows = eval_patients(model, arm, val)
        for r in rows:
            r["fold"] = f
        all_rows += rows
        pd.DataFrame(all_rows).to_csv(res_path, index=False)
        md = np.mean([r["dice"] for r in rows])
        print(f"fold {f} {arm:5s}: mean Dice {md:.3f}  ({time.time()-t0:.0f}s)")
        del model; torch.cuda.empty_cache() if DEV == "cuda" else None

## 5. Paired statistics per patient

In [ ]:
res = pd.DataFrame(all_rows)
wide = res.pivot_table(index="subject_id", columns="arm", values="dice")
vol = res.pivot_table(index="subject_id", columns="arm", values="abs_vol_err_ml")
print("Mean Dice per arm:\n", wide.mean().round(4))
print("Median abs volume error (mL):\n", vol.median().round(2))

boot_rng = np.random.default_rng(CFG["seed"] + 3)


def paired(a, b):
    d = (wide[a] - wide[b]).dropna().values
    n = len(d)
    bs = boot_rng.integers(0, n, size=(CFG["n_boot"], n))
    means = d[bs].mean(1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    sd = d.std(ddof=1)
    return {"contrast": f"{a} - {b}", "n": n, "mean_delta": d.mean(), "ci95_lo": lo, "ci95_hi": hi,
            "sd_delta": sd, "share_patients_a_better": float((d > 0).mean()),
            "MDE_at_n149 (2.8*sd/sqrt149)": 2.8 * sd / np.sqrt(149)}


contrasts = [("raw", "maps"), ("raw", "N1"), ("raw", "N2"), ("maps", "N1")]
stats = pd.DataFrame([paired(a, b) for a, b in contrasts if a in wide and b in wide]).round(4)
stats.to_csv(os.path.join(CFG["work_dir"], "paired_stats.csv"), index=False)
print(stats.to_string(index=False))

# lesion-size strata (pilot only, very small groups)
gt_ml = res.groupby("subject_id")["gt_ml"].first()
bins = pd.cut(gt_ml, [-0.1, 5, 30, 1e6], labels=["<5 mL", "5-30 mL", ">30 mL"])
print("\nMean Dice by follow-up lesion size:\n", wide.groupby(bins, observed=False).mean().round(3))

summary = {"config": CFG, "n_usable": len(SIDS), "mean_dice": wide.mean().round(4).to_dict(),
           "paired": stats.to_dict("records")}
json.dump(summary, open(os.path.join(CFG["work_dir"], "summary.json"), "w"), indent=1, default=str)

## 6. How to read the numbers (decision rules from the memo)
- `raw - maps`: if the 95% CI contains 0, the pilot shows no evidence that raw beats maps.
- `raw - N1` and `raw - N2`: if raw ≈ N1/N2, any raw advantage does not come from temporal dynamics.
- `sd_delta` is the σΔ the memo asked for; `MDE_at_n149` is the smallest Δ the full 149-case run could detect.
  If that MDE is larger than a realistic gain (for example > 0.03 Dice), the full run is unlikely to give a clear answer.
- Pilot caveats: about 40 patients, 2D U-Net (not nnU-Net), in-plane downsampling, fixed epochs, no hyperparameter search.
  The absolute Dice values are not comparable to ISLES'24 leaderboard numbers.